# Flood Time Machine: model training (Colab)

Two models: **Part A** a self-supervised change detector trained on the UTSA photos, **Part B** a river-level forecaster trained on the USGS gauges those photos link to.

## Part B context

Trains the model behind the app's **Rise** screen: given the last 72 h of river level and rain at a gauge, predict how many feet the river rises over the next 1–12 h.

* Gauges: the 18 USGS gauges the UTSA flood images are linked to (`image_metadata.csv → usgs_gage_height_station_id`).
* Held out of training entirely: **Hurricane Harvey (Aug 2017)** and the **Hill Country flood (Jul 4 2025)**. The app replays these with the model's forecasts.
* Baseline: persistence (river stays where it is). Models: LightGBM (one per lead time) + GRU (GPU).

Runtime → Change runtime type → **GPU** (T4 or better). Full run takes ~20–40 min, mostly downloading gauge data the first time (cached to Drive).

In [ ]:
# 1. Cache downloads on Drive so re-runs are fast
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['FLOOD_CACHE_DIR'] = '/content/drive/MyDrive/flood_cache'
os.makedirs(os.environ['FLOOD_CACHE_DIR'], exist_ok=True)

In [ ]:
# 2. Get the repo (needs a GitHub token with repo write access to push results at the end).
#    Add it in Colab: key icon (Secrets) → name GITHUB_TOKEN → enable notebook access.
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
REPO = 'FabianFig/Datathon_2026'
%cd /content
!rm -rf Datathon_2026
!git clone https://{token}@github.com/{REPO}.git
%cd /content/Datathon_2026
!git log --oneline -3

In [ ]:
# 3. Dependencies (pandas / numpy / torch / matplotlib are preinstalled on Colab)
!pip -q install lightgbm
# Optional: a free api.data.gov key raises USGS rate limits (https://api.waterdata.usgs.gov/signup/)
# os.environ['USGS_API_KEY'] = userdata.get('USGS_API_KEY')

## Part A: photo change detector (self-supervised, no labels)

The 575 Harvey pairs are a 2016 USDA photo and a 2017 NOAA photo of the same ground. The model learns how the 2016 camera's view maps to the 2017 camera's view across all pairs, so ordinary camera differences cancel out. What it can't predict at a location is real change on the ground (standing water, debris). Scored with 5-fold cross-validation grouped by location, so each pair is scored by a model that never saw it.

The photos are downloaded fresh into `/content` each session (fast). Extracted features are cached on Drive.

In [ ]:
# A1. Download the UTSA photos (~2 GB, 1-3 min). No need to upload anything.
!pip -q install -U huggingface_hub
!hf download jennifer-fowler/UTSA-flood-images --repo-type dataset --local-dir /content/UTSA-flood-images
!ls /content/UTSA-flood-images/images | wc -l   # expect 1204

In [ ]:
# A2. Smoke test (60 pairs, ~2 min)
!python ml/train_change_detector.py --data-dir /content/UTSA-flood-images --quick

In [ ]:
# A3. Full run (575 pairs, 5 folds x 30 epochs, ~10-15 min on a T4)
!python ml/train_change_detector.py --data-dir /content/UTSA-flood-images

In [ ]:
# A4. Results: checks vs the naive baseline, and the most-changed locations
import json, glob
from IPython.display import Image, display
m = json.load(open('models/change_detector/metrics.json'))
for k in m['checks']['learned']:
    print(f"{k:38s} learned {m['checks']['learned'][k]}   naive {m['checks']['naive'][k]}")
for p in sorted(glob.glob('ml/outputs/change_top/*.jpg'))[:6]: display(Image(p))

## Part B: river-level forecaster

In [ ]:
# 4. Smoke test (4 gauges, a few years, ~3 min). Fix anything here before the full run.
!python ml/train_stage_forecaster.py --quick --gru

In [ ]:
# 5. Full run: 18 gauges, 2014-2025, LightGBM + GRU
!python ml/train_stage_forecaster.py --gru --epochs 8

In [ ]:
# 6. Results
import json
m = json.load(open('models/stage_forecaster/metrics.json'))
print('Selected:', m['selected'], '| rows', m['rows'])
for model, res in m['results'].items():
    for split, r in res.items():
        print(f"{model:11s} {split:7s} RMSE@1/3/6h", [round(r['by_lead'][str(h)]['rmse'],2) for h in (1,3,6)],
              'persistence', [round(r['by_lead'][str(h)]['rmse_persistence'],2) for h in (1,3,6)],
              'CSI', round(r['csi_rise_mean'],2))
print('Top features (6h):', m['feature_importance_6h'])
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('ml/outputs/*.png')): display(Image(p))

Optional: re-run with `--no-future-rain` to show how much of the skill comes from knowing the rainfall forecast. Report both in the slides; the app uses the default (rainfall-forecast-assumed) model and says so.

In [ ]:
# 7. Push results so the app picks them up (teammates: git pull, then reload Expo Go)
!git config user.name "Colab trainer"
!git config user.email "colab@users.noreply.github.com"
!git add models/stage_forecaster models/change_detector ml/outputs mobile-app/src/data/floodScenarios.json mobile-app/src/data/changeScores.json
!git commit -m "Train change detector and stage forecaster on Colab"
!git push origin HEAD